# 02 · Curvas, subajuste y depuración


Ejecuta las celdas en orden. Antes de mirar la solución, intenta la actividad y anota una predicción. Si cambias parámetros, vuelve a ejecutar desde la celda de creación del modelo: llamar a `fit` otra vez continúa el entrenamiento anterior.

**Entorno:** Google Colab o Jupyter con `pip install tensorflow numpy matplotlib scikit-learn notebook`. Reinicia el kernel tras instalar si es necesario. No se necesitan descargas de conjuntos de datos. Los resultados y tiempos pueden variar por versión, CPU y aleatoriedad.

**Tiempo:** 30–35 minutos.

## 1. Cómo leer una curva
- **Subajuste:** entrenamiento y validación muestran error alto respecto a un referente razonable; puede haber poca capacidad, optimización insuficiente o características inadecuadas.
- **Sobreajuste:** el desempeño de entrenamiento mejora mientras el de validación deja de mejorar o empeora; el aumento de la brecha es una señal, no una prueba por sí sola.
- **Depuración:** verifica formas, escalas, rangos, pérdida y salida compatibles, presencia de NaN y aprendizaje en un lote pequeño. No uses prueba para ajustar.

Las pérdidas de entrenamiento y validación se calculan de maneras algo distintas durante `fit`; pequeñas diferencias, fluctuaciones y regularización no bastan para diagnosticar.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)

In [ ]:
def curvas(h, metrica="loss"):
    plt.figure(figsize=(7, 4))
    plt.plot(h.history[metrica], label="Entrenamiento")
    plt.plot(h.history["val_" + metrica], label="Validación")
    plt.xlabel("Época")
    plt.ylabel(metrica)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

## 2. Genera una relación curva y separa los datos

In [ ]:
rng = np.random.default_rng(42)
X = rng.uniform(-2, 2, (600, 1)).astype("float32")
y = (X[:, 0] ** 2 + rng.normal(0, 0.15, 600)).astype("float32")
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
plt.scatter(X_train[:, 0], y_train, alpha=0.4, s=10)
plt.xlabel("x")
plt.ylabel("y")
plt.show()
print("Formas:", X_train.shape, y_train.shape)
print("Valores finitos:", np.isfinite(X_train).all(), np.isfinite(y_train).all())

## 3. Una recta frente a una curva

Predice cuál modelo tendrá mayor error y por qué. Ambas configuraciones entrenan con los mismos datos.

In [ ]:
def construir(tipo):
    keras.utils.set_random_seed(42)
    capas = [keras.Input(shape=(1,))]
    if tipo == "no_lineal":
        capas += [
            keras.layers.Dense(16, activation="relu"),
            keras.layers.Dense(16, activation="relu"),
        ]
    capas += [keras.layers.Dense(1)]
    modelo = keras.Sequential(capas)
    modelo.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse", metrics=["mae"])
    return modelo


modelos, historias = {}, {}
for tipo in ["lineal", "no_lineal"]:
    modelos[tipo] = construir(tipo)
    historias[tipo] = modelos[tipo].fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        verbose=0,
    )
    print(
        tipo, "MAE val:", round(modelos[tipo].evaluate(X_val, y_val, verbose=0)[1], 3)
    )
    curvas(historias[tipo])

In [ ]:
orden = np.argsort(X_val[:, 0])
plt.scatter(X_val[:, 0], y_val, s=10, alpha=0.35, label="Observado")
for tipo, modelo in modelos.items():
    pred = modelo.predict(X_val, verbose=0).ravel()
    plt.plot(X_val[orden, 0], pred[orden], label=tipo)
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 4. Tu turno: depura una incompatibilidad

Un compañero quiere predecir una cantidad continua, pero usa `Dense(1, activation="sigmoid")` como salida. ¿Qué restricción introduce? Cambia solo la salida y compara con el modelo no lineal anterior. Revisa rango de `y_train` antes de entrenar.

In [ ]:
print("Rango del objetivo:", y_train.min(), y_train.max())
# Tu código aquí.

**Mi diagnóstico:** escribe aquí.

## 5. Solución de referencia

In [ ]:
keras.utils.set_random_seed(42)
limitado = keras.Sequential(
    [
        keras.Input(shape=(1,)),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ]
)
limitado.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse", metrics=["mae"])
h_limitado = limitado.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    verbose=0,
)
print("MAE validación, sigmoide:", limitado.evaluate(X_val, y_val, verbose=0)[1])
print(
    "MAE validación, salida lineal:",
    modelos["no_lineal"].evaluate(X_val, y_val, verbose=0)[1],
)
curvas(h_limitado)

Sigmoide limita la salida a (0,1), pero el objetivo llega aproximadamente a 4. Una salida lineal no impone ese límite. Antes de aumentar épocas conviene revisar la compatibilidad entre objetivo y salida.

## 6. Desafío opcional

Entrena con 2, 20 y 100 épocas desde modelos nuevos. Compara curvas, predicciones y MAE de validación; distingue falta de entrenamiento de falta de capacidad. El sobreajuste se examina de forma controlada en el notebook 04.